In [1]:
import pandas as pd
import numpy as np

In [2]:
from bs4 import BeautifulSoup
import requests
import logging
import requests

from requests.adapters import HTTPAdapter, Retry

s = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[ 502, 503, 504 ])
s.mount('http://', HTTPAdapter(max_retries=retries))


In [3]:
mapped_ids = pd.read_csv('mapped_ids.csv')
mapped_ids

,Unnamed: 0,CCLE,clean_cellosaurus
0,0,HCC827_LUNG,HCC827
1,0,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HL-60
2,0,CORL311_LUNG,COR-L311
3,0,NCIH889_LUNG,NCI-H889
4,0,NCIH2029_LUNG,NCI-H2029
...,...,...,...
1828,0,PL18_PANCREAS,Panc 05.04
1829,0,TK_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,TK [Human B-cell lymphoma]
1830,0,MOT_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Mo
1831,0,CCLFUPGI0005T_STOMACH,CCLF-UPGI-0005-T


In [4]:
vendor_ids = pd.DataFrame(columns=['CCLE', 'clean_cellosaurus', 'ATCC_id', 'DSMZ_id'])
vendor_ids

,CCLE,clean_cellosaurus,ATCC_id,DSMZ_id


# Overview
This workflow takes an input dataframe called **mapped_ids** with the columns **'CCLE'** (Cancer Cell Line Encyclopedia ID) and **'clean_cellosaurus'** (cell lines cellosaurus id).  Depending on intended scraping purposes, the workflow can be easily modified -- the essential column is the 'clean_cellosaurus' column with the cellosaurus ids.  

**For any input dataframe with a column of cellosaurus IDs, this workflow will:**
1. Scrape matching cell line pages from cellosaurus for ATCC and DSMZ vendor ids.
2. Using scraped vendor ids, visit matching vendor pages and scrape any available karyotype information.

**Note**: modify the line below the comment to support different input DataFrame columns

In [5]:
for index, row in mapped_ids.iterrows():
    search_URL = "https://www.cellosaurus.org/search?query=%s"%row['clean_cellosaurus']
    soup = BeautifulSoup(requests.get(search_URL).text, 'lxml')

    cellosaurus_id = soup.find_all('td')[0].text
    cell_URL = "https://www.cellosaurus.org/%s"%cellosaurus_id
    soup = BeautifulSoup(requests.get(cell_URL).text, 'lxml')

    text = soup.text
    if 'Cell line collections' not in text:
        continue
    text = text[text.index('Cell line collections'):text.index('Cell line collections')+1000]

    ATCC = np.nan
    DSMZ = np.nan

    try:
        if 'ATCC' in text:
            ATCC_index = text.index('ATCC')
            ATCC = text[ATCC_index+6:ATCC_index + text[ATCC_index+6:].index(" ")].strip()

        if 'DSMZ' in text:
            DSMZ_index = text.index('DSMZ')
            DSMZ = text[DSMZ_index+6:DSMZ_index + text[DSMZ_index+6:].index(" ")].strip()
    except Exception as e:
        print(e)
    
    #Depending on the shape and column names of the input dataframe, modifying this line may be necessary
    vendor_ids = pd.concat([vendor_ids, pd.DataFrame([{'CCLE':row['CCLE'], 'clean_cellosaurus':row['clean_cellosaurus'], 'ATCC_id':ATCC, 'DSMZ_id':DSMZ}])])

vendor_ids

,CCLE,clean_cellosaurus,ATCC_id,DSMZ_id
0,HCC827_LUNG,HCC827,CRL-2868,ACC-566
0,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HL-60,CCL-240,ACC-3
0,CORL311_LUNG,COR-L311,NaN,NaN
0,NCIH889_LUNG,NCI-H889,CRL-5817,NaN
0,NCIH2029_LUNG,NCI-H2029,CRL-5913,NaN
...,...,...,...,...
0,LC2AD_LUNG,LC-2/ad,NaN,NaN
0,PC3JPC3_LUNG,PC-3 [Human lung carcinoma],NaN,NaN
0,PL18_PANCREAS,Panc 05.04,CRL-2557,NaN
0,TK_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,TK [Human B-cell lymphoma],NaN,NaN


In [6]:
import time
headers = {'user-agent': 'Mozilla/5.0'}

def get_karyotype_atcc(cell_id):
    soup = ''
    while soup == '':
        try:
            URL = "https://www.atcc.org/products/%s#detailed-product-information/"%cell_id
            soup = BeautifulSoup(requests.get(URL, headers=headers).text, 'lxml')
        except:
            time.sleep(5)
    
    text_to_mine = soup.text.split('\n')
    
    if 'Karyotype' in text_to_mine:
        ind = text_to_mine.index("Karyotype")

        final = ''
        for i in range(ind, ind+3):
            text = text_to_mine[i]
            if text !='' and len(text.split(' '))>1:
                final = text.strip()
    else:
        final = 'none'
            
    return(final)

def get_karyotype_dsmz(cell_id):
    URL= "https://www.dsmz.de/collection/catalogue/details/culture/%s"%cell_id
    soup =''
    while soup == '':
        try:
            soup = BeautifulSoup(requests.get(URL, headers=headers).text, 'lxml')
        except:
            time.sleep(5)

    text_to_mine = soup.text.split('\n')

    final = ''
    for i in range(0, len(text_to_mine)):
        if 'Cytogenetics' in text_to_mine[i]:
            final = text_to_mine[i+1]      

    return(final.strip())

In [7]:
karyotypes = pd.DataFrame()

for index, row in vendor_ids.iterrows():
    ATCC_Karyotype = ''
    if row['ATCC_id'] != '':
        ATCC_Karyotype = get_karyotype_atcc(row['ATCC_id'])
    DSMZ_Karyotype = ''
    if row['DSMZ_id'] != '':
        DSMZ_Karyotype = get_karyotype_dsmz(row['DSMZ_id'])

    karyotypes = pd.concat([karyotypes, pd.DataFrame([{'CCLE':row['CCLE'],'clean_cellosaurus':row['clean_cellosaurus'], 'ATCC':row['ATCC_id'], 'DSMZ':row['DSMZ_id'], 'ATCC_Karyotype': ATCC_Karyotype, 'DSMZ_Karyotype': DSMZ_Karyotype}])])
karyotypes

,CCLE,clean_cellosaurus,ATCC,DSMZ,ATCC_Karyotype,DSMZ_Karyotype
0,HCC827_LUNG,HCC827,CRL-2868,ACC-566,none,human flat-moded hypotriploid karyotype with 6...
0,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HL-60,CCL-240,ACC-3,The stemline chromosome number is pseudodiploi...,human flat-moded hypotetraploid karyotype with...
0,CORL311_LUNG,COR-L311,NaN,NaN,none,
0,NCIH889_LUNG,NCI-H889,CRL-5817,NaN,none,
0,NCIH2029_LUNG,NCI-H2029,CRL-5913,NaN,none,
...,...,...,...,...,...,...
0,LC2AD_LUNG,LC-2/ad,NaN,NaN,none,
0,PC3JPC3_LUNG,PC-3 [Human lung carcinoma],NaN,NaN,none,
0,PL18_PANCREAS,Panc 05.04,CRL-2557,NaN,none,
0,TK_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,TK [Human B-cell lymphoma],NaN,NaN,none,


## Counts of karyotypes found from each vendor

In [20]:
np.count_nonzero(karyotypes['ATCC_Karyotype']!='none')

290

In [21]:
np.count_nonzero(karyotypes['DSMZ_Karyotype']!='')

354